# The t-Statistic: Testing When $\sigma$ Is Unknown

Every test so far borrowed a luxury: a known population standard deviation, making $(\bar{X} - \mu)/(\sigma/\sqrt{n})$ a standard normal — the **z-statistic**. Give the luxury up (the common case) and the natural move is to substitute the sample standard deviation $s$ — producing the **t-statistic**, which is *not* standard normal: it follows the **Student t-distribution** from the confidence-intervals lesson, bell-shaped but with **heavier tails** that price in the extra uncertainty of estimating $\sigma$. Its single parameter, the **degrees of freedom** $\nu = n - 1$, depends only on the sample size — $\nu = 9$ for the ten heights — and as $\nu$ approaches 30 the t-curve becomes almost indistinguishable from the Gaussian, which is exactly where the "30 samples" rule of thumb comes from. It is based on the eighth video of Lesson 8 (Hypothesis Testing).

## 1. Introduction

The t-distribution already rescued confidence intervals (Lesson 7); this notebook re-imports it into the four-step testing pipeline. Only step 2 changes — the statistic and its null distribution — but that change moves every critical value and every p-value, and ignoring it quietly corrupts $\alpha$.

After completing this notebook you will be able to:

- Write the t-statistic $(\bar{X} - \mu)/(S/\sqrt{n})$ and say exactly which ingredient changed from the z-statistic.
- State its null distribution — Student t with $\nu = n - 1$ degrees of freedom — and what the heavier tails compensate for.
- Explain why $\nu$ depends only on $n$ (never on $\mu$ or $\sigma$) and why $\nu \approx 30$ makes t and Gaussian nearly identical.
- Compute the observed t-statistic for the heights data and see how it differs from the known-$\sigma$ z-statistic.
- Demonstrate by simulation what goes wrong if you use z critical values with an estimated $s$.

## 2. Theory

### The known-$\sigma$ world, one more time

Heights are modeled as Gaussian with parameters $\mu$ and $\sigma^2$, so with $n = 10$ samples the sample mean is Gaussian too, with the same mean and standard deviation $\sigma/\sqrt{10}$. If both parameters are known, standardizing gives the familiar

$$
Z = \frac{\bar{X} - \mu}{\sigma/\sqrt{n}} \sim N(0, 1) \qquad \text{— the z-statistic.}
$$

### Losing $\sigma$

When $\sigma$ is unknown — the more common case — knowing "$\bar{X}$ is Gaussian with standard deviation $\sigma/\sqrt{10}$" is not very useful: you can't evaluate it. The fix: replace $\sigma$ in the standardization by its estimate, the sample standard deviation

$$
S = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(X_i - \bar{X})^2}
$$

(almost the sample variance's square root, except dividing by $n - 1$ instead of $n$). The result is the **t-statistic**:

$$
T = \frac{\bar{X} - \mu}{S/\sqrt{n}}
$$

### Its distribution — sadly, not normal

$T$ follows the **Student t-distribution**: bell-shaped and similar to the Gaussian, but with **heavier tails**. The extra tail probability accounts for the uncertainty introduced when the population standard deviation is replaced by an estimate — $S$ is itself random, and samples that happen to underestimate $\sigma$ inflate $T$ (the mechanism dissected in Lesson 7's t-notebook).

### One parameter: degrees of freedom

The t-distribution has a single parameter, the **degrees of freedom**, written $\nu$: $X \sim t_\nu$. It controls how heavy the tails are — small $\nu$, heavy tails; and as $\nu$ grows toward 30, the t-PDF and the Gaussian PDF look almost exactly alike. *That* is why statisticians like having 30 samples.

For a sample of size $n$:

$$
\nu = n - 1
$$

— for the ten heights, $\nu = 9$. Note what $\nu$ does **not** depend on: the population mean and variance. Only the number of samples you collected matters, which also means that as $n$ increases, the t-statistic's distribution turns more and more Gaussian.

### Where this plugs into the pipeline

In the four steps, only step 2's design changes: the statistic is $T$ instead of $Z$, and its null distribution is $t_{n-1}$ instead of $N(0,1)$. Critical values and p-values are then read off the t-distribution — which, having fatter tails, demands slightly more extreme evidence. **In summary: the t-statistic is used when the population is Gaussian with unknown $\sigma$.**

## 3. Worked Examples

### Example 1: the observed t-statistic for the heights

The ten heights have $\bar{x} = 68.442$ and $s = 3.113$ (computed with the $n-1$ divisor). Under $H_0: \mu = 66.7$:

$$
t_{obs} = \frac{68.442 - 66.7}{3.113/\sqrt{10}} = \frac{1.742}{0.984} = 1.770
$$

Compare the known-$\sigma$ z-statistic from notebook 04: $z_{obs} = 1.742/0.949 = 1.836$. The two *numbers* are close here, because this sample's $s = 3.11$ happens to sit near the assumed $\sigma = 3$ — the decisive difference is the **curve** each must be read on. On the fat-tailed $t_9$, the same-sized statistic buys less evidence, and (as the next notebook makes the headline) the verdict actually flips.

### Example 2: degrees-of-freedom bookkeeping

$n = 10 \Rightarrow \nu = 9$; $n = 25 \Rightarrow \nu = 24$; $n = 200 \Rightarrow \nu = 199$ (where t and z are practically the same curve). Always $n - 1$; never a function of $\mu$ or $\sigma$.

### Example 3: what the heavier tails change

95th-percentile critical values for a right-tailed test: $z_{0.95} = 1.645$ but $t_{0.95,\,9} = 1.833$. The t-test demands a *larger* observed statistic for the same $\alpha$ — the honest surcharge for not knowing $\sigma$. Using 1.645 with an estimated $s$ would reject too eagerly (Exercise 5 measures exactly how much).

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project). As in Lesson 7, the t-distribution is built from its density — critical values and tail areas by numeric integration, no statistics library required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from math import gamma, sqrt, pi

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(88)

def t_pdf(t, df):
    """Density of the Student t-distribution with df degrees of freedom."""
    coeff = gamma((df + 1) / 2) / (sqrt(df * pi) * gamma(df / 2))
    return coeff * (1 + np.asarray(t, dtype=float) ** 2 / df) ** (-(df + 1) / 2)

def t_tail(t_value, df):
    """P(T > t_value) for T ~ t_df, by numeric integration."""
    grid = np.linspace(0, abs(t_value), 20_000)
    cdf_half = np.trapezoid(t_pdf(grid, df), grid)
    tail = 0.5 - cdf_half
    return tail if t_value >= 0 else 1 - tail

def t_quantile(p, df):
    lo, hi = 0.0, 700.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if 1 - t_tail(mid, df) < p:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

Phi = lambda z: 0.5 * (1 + float(sp.erf(sp.Float(z) / sp.sqrt(2))))

### The observed t-statistic for the heights

In [ ]:
heights = np.array([64.12, 71.15, 66.41, 70.17, 73.13,
                    65.10, 69.16, 67.39, 71.97, 65.82])
MU0 = 66.7
n = len(heights)
x_bar = heights.mean()
s = heights.std(ddof=1)                      # the n - 1 divisor

t_obs = (x_bar - MU0) / (s / np.sqrt(n))
z_obs_known_sigma = (x_bar - MU0) / (3.0 / np.sqrt(n))

print(f"x-bar = {x_bar:.3f}, s = {s:.3f}, degrees of freedom nu = {n - 1}")
print(f"t-statistic (sigma estimated by s): {t_obs:.3f}")
print(f"z-statistic (sigma = 3 known):      {z_obs_known_sigma:.3f}")
print("\nWith s = 3.113 landing close to the assumed sigma = 3, the two statistics")
print("nearly agree -- what changes the game is the CURVE each is read on.")

### Critical values and the p-value, on the right curve

The t-test's threshold sits farther out than z's — and this time, the observation does **not** clear it.

In [ ]:
alpha = 0.05
t_crit = t_quantile(1 - alpha, n - 1)
z_crit = float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(1 - alpha) - 1))
print(f"right-tailed critical values at alpha = 0.05: t_(0.95, 9) = {t_crit:.3f} "
      f"vs z_0.95 = {z_crit:.3f}")

p_t = t_tail(t_obs, n - 1)
p_wrong = 1 - Phi(t_obs)
print(f"\np-value of t_obs = {t_obs:.3f} on the t_9 curve:     {p_t:.4f}")
print(f"(what the normal curve would wrongly claim: {p_wrong:.4f})")
print(f"\nverdict: p = {p_t:.4f} >= 0.05 -> FAIL to reject H0. The known-sigma")
print("z-test rejected this very sample (p = 0.0332); pricing in the estimated")
print("sigma withdraws the rejection -- the next notebook runs all three tests")
print("and makes this flip the headline.")

### The t-statistic really is t-distributed — a simulation witness

Draw thousands of 10-person samples from a Gaussian, form each one's t-statistic, and lay the histogram over both candidate curves.

In [ ]:
samples = rng.normal(66.7, 3.0, size=(60_000, n))
t_stats = ((samples.mean(axis=1) - 66.7)
           / (samples.std(axis=1, ddof=1) / np.sqrt(n)))

t_axis = np.linspace(-5, 5, 500)
plt.figure(figsize=(10, 4.8))
plt.hist(t_stats, bins=160, range=(-5, 5), density=True,
         color="tab:green", alpha=0.5, label="simulated t-statistics (n = 10)")
plt.plot(t_axis, t_pdf(t_axis, n - 1), color="tab:red", linewidth=2,
         label=r"$t_9$ density")
plt.plot(t_axis, np.exp(-0.5 * t_axis**2) / np.sqrt(2 * np.pi), color="black",
         linewidth=1.5, linestyle="--", label="standard normal")
plt.xlabel("t")
plt.ylabel("density")
plt.title("The statistic follows the t-curve, not the normal -- the tails give it away")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

print(f"fraction of |T| > 3: simulated {np.mean(np.abs(t_stats) > 3):.4f}, "
      f"t_9 predicts {2 * t_tail(3, 9):.4f}, normal would predict "
      f"{2 * (1 - Phi(3)):.4f}")

## 5. Visualizations

### Degrees of freedom: the march toward the Gaussian

The video's picture — by $\nu \approx 30$, the two curves are nearly indistinguishable, which is precisely the origin of the "30 samples" rule.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(t_axis, np.exp(-0.5 * t_axis**2) / np.sqrt(2 * np.pi), color="black",
         linewidth=2.5, label="standard normal")
for df, color in [(1, "tab:red"), (5, "tab:orange"), (9, "tab:green"),
                  (30, "tab:blue")]:
    plt.plot(t_axis, t_pdf(t_axis, df), color=color, linewidth=1.6,
             label=f"$t_{{{df}}}$")
plt.xlabel("t")
plt.ylabel("density")
plt.title(r"One parameter, $\nu$: heavier tails at small $\nu$, Gaussian by $\nu \approx 30$")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

### The test, drawn on the correct curve

The $t_9$ null distribution with its $\alpha = 0.05$ critical region, the z threshold that would have been (too lenient), and the observed 2.893.

In [ ]:
plt.figure(figsize=(10, 4.8))
plt.plot(t_axis, t_pdf(t_axis, 9), color="black", linewidth=1.8,
         label=r"null distribution of $T$: $t_9$")
plt.fill_between(t_axis, t_pdf(t_axis, 9), where=t_axis >= t_crit,
                 color="tab:red", alpha=0.5,
                 label=f"reject beyond $t_{{0.95,9}}$ = {t_crit:.3f}")
plt.axvline(z_crit, color="tab:orange", linestyle="--", linewidth=1.5,
            label=f"z threshold {z_crit:.3f} (too lenient here)")
plt.axvline(t_obs, color="tab:blue", linewidth=2.2,
            label=f"observed t = {t_obs:.3f}")
plt.xlim(-4.5, 4.5)
plt.xlabel("t")
plt.ylabel("density")
plt.title("The t-test for the heights: past the z bar, short of the honest t bar")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** State the degrees of freedom for samples of size 5, 25, and 200, and say for which of the three the normal approximation to the t-distribution is already respectable.

**Exercise 2.** For each scenario, z-statistic or t-statistic? (a) Gaussian population, $\sigma$ known from decades of factory records, $n = 8$; (b) Gaussian population, $\sigma$ unknown, $n = 12$; (c) unknown population shape, $\sigma$ unknown, $n = 500$.

### Intermediate

**Exercise 3.** A different sample of 10 heights has the same $\bar{x} = 68.442$ but a tighter $s = 1.9$. Compute its t-statistic, p-value, and verdict at $\alpha = 0.05$ — and contrast with the worked example ($s = 3.113$, fail to reject) to show how $s$ alone can flip the conclusion.

**Exercise 4.** Compare $P(T_9 > 2)$ with $P(Z > 2)$. Which is larger, by how much, and what does that mean for p-values computed on the wrong curve?

### Advanced

**Exercise 5.** *The cost of the wrong curve.* Simulate 100,000 ten-person samples with $H_0$ true, form the t-statistics, and apply the *z* critical value 1.645. What Type I rate do you actually get instead of 5%? Repeat at $n = 5$ and $n = 30$ and summarize when the z shortcut becomes tolerable.

**Exercise 6.** *The pivot property.* Simulate t-statistics from three very different Gaussian populations — $(\mu, \sigma) = (66.7, 3)$, $(0, 1)$, and $(1000, 250)$ — each with $n = 10$, computing each statistic around its own true mean. Overlay the three histograms and explain why they coincide, and why that coincidence (with $\nu$ depending on $n$ alone) is what makes one t-table serve every problem.

## 7. Solutions

In [ ]:
# Exercise 1: degrees of freedom
for n_ in (5, 25, 200):
    print(f"n = {n_:>3d} -> nu = {n_ - 1}")
print("Only n = 200 (nu = 199) is comfortably in normal-approximation territory;")
print("nu = 24 is close but still visibly fatter-tailed; nu = 4 is nowhere near.")

In [ ]:
# Exercise 2: which statistic
print("(a) z-statistic -- sigma genuinely known; n is irrelevant to the CHOICE.")
print("(b) t-statistic -- Gaussian population, sigma estimated: the textbook case.")
print("(c) t formally (sigma unknown), but at n = 500 the t and z versions are")
print("    indistinguishable, and the CLT covers the unknown population shape.")

In [ ]:
# Exercise 3: same mean, tighter sample
s3 = 1.9
t3 = (68.442 - MU0) / (s3 / np.sqrt(10))
p3 = t_tail(t3, 9)
print(f"t = (68.442 - 66.7)/(1.9/sqrt(10)) = {t3:.3f}, p = {p3:.4f}")
print(f"verdict at alpha = 0.05: {'REJECT' if p3 < 0.05 else 'fail to reject'}")
print("\nSame x-bar as the worked example, but the tighter sample (s = 1.9 vs 3.11)")
print("turns the identical 1.742-inch gap into strong evidence -- with estimated")
print("sigma, the conclusion belongs to the pair (x-bar, s), not x-bar alone.")

In [ ]:
# Exercise 4: tail comparison at 2
pt = t_tail(2, 9)
pz = 1 - Phi(2)
print(f"P(T_9 > 2) = {pt:.4f}   P(Z > 2) = {pz:.4f}   ratio = {pt / pz:.2f}x")
print("The t tail is ~1.6x heavier at 2: p-values computed on the normal curve")
print("understate the true p, overselling the evidence against H0.")

In [ ]:
# Exercise 5: alpha inflation from the z shortcut
for n_ in (5, 10, 30):
    sims = rng.normal(0, 1, size=(100_000, n_))
    ts = sims.mean(axis=1) / (sims.std(axis=1, ddof=1) / np.sqrt(n_))
    realized_alpha = (ts > 1.645).mean()
    print(f"n = {n_:>2d}: intended alpha 5.0%, realized with z threshold: "
          f"{realized_alpha:.1%}")
print("\nThe shortcut inflates Type I errors most where samples are smallest;")
print("by n = 30 the inflation is a fraction of a point -- the rule of thumb again.")

In [ ]:
# Exercise 6: the pivot property
plt.figure(figsize=(10, 4.4))
for (mu_, sig_), color in [((66.7, 3.0), "tab:green"),
                           ((0.0, 1.0), "tab:blue"),
                           ((1000.0, 250.0), "tab:red")]:
    sims = rng.normal(mu_, sig_, size=(40_000, 10))
    ts = (sims.mean(axis=1) - mu_) / (sims.std(axis=1, ddof=1) / np.sqrt(10))
    plt.hist(ts, bins=120, range=(-5, 5), density=True, histtype="step",
             linewidth=1.6, color=color,
             label=f"population N({mu_:g}, {sig_:g}$^2$)")
plt.plot(t_axis, t_pdf(t_axis, 9), color="black", linewidth=1.2, linestyle="--",
         label=r"$t_9$")
plt.xlabel("t-statistic")
plt.ylabel("density")
plt.title("Exercise 6: three wildly different populations, one t-distribution")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

print("Standardizing by the sample's own x-bar and s cancels mu and sigma exactly,")
print("leaving a distribution that depends on n alone (nu = n - 1). That pivot")
print("property is why a single t-table -- indexed only by degrees of freedom --")
print("serves every Gaussian testing problem ever posed.")

## 8. Conclusion

- With $\sigma$ unknown, standardize with $S$ (the $n-1$ divisor) instead: the **t-statistic** $T = (\bar{X} - \mu)/(S/\sqrt{n})$ — used when the population is Gaussian with unknown $\sigma$.
- $T$ is *not* standard normal: it follows the **Student t-distribution**, whose heavier tails price the randomness of $S$.
- One parameter, the **degrees of freedom $\nu = n - 1$** — for the ten heights, $\nu = 9$ — depending only on sample size, never on $\mu$ or $\sigma$ (the pivot property behind every t-table); by $\nu \approx 30$ the t and Gaussian curves nearly coincide, the origin of the "30 samples" rule.
- For the heights: $t_{obs} = 1.770$ — numerically close to the known-$\sigma$ $z_{obs} = 1.836$, but read on the fat-tailed $t_9$ its $p = 0.055$ no longer clears $\alpha = 0.05$: the honest test withdraws the rejection the z-test made.
- The warning, quantified: z critical values with an estimated $s$ inflate the true Type I rate (nearly 9% instead of 5% at $n = 5$) — match the curve to the statistic.

Next: running full hypothesis tests with the t-machinery — and from there, the lesson's destination: comparing *two* groups, A/B testing.